###**1. Import Library**

In [89]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

##**1. Load Data & Labeling**

In [90]:
df21 = pd.read_excel('2021.xlsx')
df22 = pd.read_excel('2022.xlsx')
df23 = pd.read_excel('2023.xlsx')
df24 = pd.read_excel('2024.xlsx')

In [91]:
# Tambahkan kolom tahun sebelum digabung
df21['Year'] = 2021
df22['Year'] = 2022
df23['Year'] = 2023
df24['Year'] = 2024

##**2. Merge Data**

In [92]:
df = pd.concat([df21, df22, df23, df24], ignore_index=True)

##**3. Rename Columns & Cleanup**

In [93]:
df = df.drop(columns=['No.'], errors='ignore') # Hapus kolom No.
df.columns = [
    'gender', 'age_month', 'weight', 'height',
    'status_w_a', 'zscore_w_a',
    'status_h_a', 'zscore_h_a',
    'status_w_h', 'zscore_w_h', 'year'
]

##**4. Deep Cleaning (Convert to Numeric)**

In [94]:
# Paksa semua kolom numerik menjadi float
num_cols = ['age_month', 'weight', 'height', 'zscore_w_a', 'zscore_h_a', 'zscore_w_h']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

##**5. Text Standardization**

In [95]:
# Satukan variasi kategori sebelum encoding
df['status_h_a'] = df['status_h_a'].replace({'Sangat Pendek': 'Stunted', 'Pendek': 'Stunted'})

##**6. Encoding**

In [96]:
df['gender'] = df['gender'].map({'F': 0, 'M': 1, 'M ': 1, 'F ': 0}) # Menangani spasi jika ada
mapping_status = {
    'Normal': 0, 'Underfed': 1, 'Malnutrition': 2, 'Severely Malnutrition': 3,
    'Not Stunted': 0, 'Stunted': 1, 'Severely Stunted': 2,
    'Thin': 1, 'Severely Thin': 2, 'Obese': 3
}
for col in ['status_w_a', 'status_h_a', 'status_w_h']:
    df[col] = df[col].map(mapping_status).fillna(0).astype(int)

##**7. Handling Missing & Outliers**

In [97]:
df = df.dropna() # Hapus nilai NaN hasil konversi
df = df.drop_duplicates()

In [98]:
# Hapus Outlier
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    return df[(df[column] >= (Q1 - 1.5 * IQR)) & (df[column] <= (Q3 + 1.5 * IQR))]

df = remove_outliers(df, 'weight')
df = remove_outliers(df, 'height')

##**8. Scaling**

In [99]:
features = ['age_month', 'weight', 'height', 'zscore_w_a', 'zscore_h_a', 'zscore_w_h']
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

##**9. Final Output**

In [100]:
print("Data Wrangling Berhasil!")
display(df_scaled.head())
df_scaled.to_csv('Data_Stunting_Siap_Model.csv', index=False)

Data Wrangling Berhasil!


,gender,age_month,weight,height,status_w_a,zscore_w_a,status_h_a,zscore_h_a,status_w_h,zscore_w_h,year
0,0,1.360315,0.844346,1.132247,0,-0.345776,1,0.057486,0,-0.351464,2021
1,1,0.698587,0.352093,0.629179,0,-0.324111,1,-0.039094,0,-0.287207,2021
2,1,1.558834,1.172514,1.086514,0,-0.302447,1,-0.355172,0,0.079977,2021
3,1,-0.492525,-0.058118,-0.559891,0,0.509968,1,-0.820508,0,1.144812,2021
4,0,1.691179,1.418641,1.177981,0,-0.042474,1,-0.276152,0,0.355365,2021
